In [1]:
from pathlib import Path
import pandas as pd

import pickle
DATA_DIR = Path("/root/capsule/data")
SCRATCH_DIR = Path("/root/capsule/scratch")
MATCH_DIR = SCRATCH_DIR / "ophys_xenium_match_tables"
MATCH_DIR.mkdir(exist_ok=True)

load data information

In [2]:
data_info = pickle.load(open('/root/capsule/code/data_info.pkl', 'rb'))
mouse_ids = list(data_info.keys())
print('mouse_ids:', mouse_ids)

FileNotFoundError: [Errno 2] No such file or directory: '/root/capsule/code/data_info.pkl'

Find same Neurons across three drifting grating sessions

In [ ]:

    
for mouse_id in mouse_ids:
    ophys_session_names = {asset_name: data_info[mouse_id]['ophys'][asset_name]['raw'] for asset_name in data_info[mouse_id]['ophys'] if data_info[mouse_id]['ophys'][asset_name]['session_type'] == 'STAGE_1'}

    ophys_zstack_path = DATA_DIR / data_info[mouse_id]['coregistration']['ophys_zstack']
    ophys_zstack_path = ophys_zstack_path / list(ophys_zstack_path.glob("*"))[0]
    planes = [f"VISp_{i}" for i in range(8)]
    ophys_zstack_match = []
    for plane in planes:
        for session, session_name in ophys_session_names.items():
            cell_matching_path = ophys_zstack_path / session_name / 'cell_matching' / f"{plane}_to_zstack_match.csv"
            df_match = pd.read_csv(cell_matching_path)[['mask_id_fov','mask_id_cz','plane','mouse_id','x_cz','y_cz','z_cz']]
            df_match.rename(columns={'mask_id_fov': f'cell_id - {session}'}, inplace=True)
            if session == "session_1":
                df_plane = df_match.copy()
            else:
                df_plane = pd.merge(df_plane, df_match, on=['mask_id_cz','plane','mouse_id','x_cz','y_cz','z_cz'], how='left')
        # After merging all sessions, drop rows with any NaN cell_id and cast to int
        for session in ophys_session_names:
            df_plane.dropna(subset=[f'cell_id - {session}'], inplace=True)
            df_plane[f'cell_id - {session}'] = df_plane[f'cell_id - {session}'].astype(int)
        ophys_zstack_match.append(df_plane)

    ophys_zstack_match = pd.concat(ophys_zstack_match,axis= 0)
    ophys_zstack_match.rename(columns={'mask_id_cz': 'cell_id - zstack', 'x_cz': 'x_loc', 'y_cz': 'y_loc', 'z_cz': 'z_loc'}, inplace=True)
    ophys_zstack_match = ophys_zstack_match[['mouse_id', 'plane', 'cell_id - zstack', 'x_loc', 'y_loc', 'z_loc', 'cell_id - session_1', 'cell_id - session_2', 'cell_id - session_3']]

    xenium_zstack_match = []
    xenium_zstack_path = DATA_DIR / data_info[mouse_id]['coregistration']['xenium_zstack'] / "cell_matching"
    cell_matching_files =   list(xenium_zstack_path.glob("*.csv"))
    for file in cell_matching_files:
        df = pd.read_csv(file)
        df['xenium_section'] = int(file.stem.split('_')[1])
        df['mouse_id'] = mouse_id
        mask_cz_counts = df.groupby('mask_id_cz')['mask_id_xenium'].nunique()
        df = df[df['iou'] > 0.4]
        duplicate_mask_cz = mask_cz_counts[mask_cz_counts > 1].index
        df = df[~df['mask_id_cz'].isin(duplicate_mask_cz)]
        xenium_zstack_match.append(df)
    
    xenium_zstack_match = pd.concat(xenium_zstack_match, ignore_index=True)
    xenium_zstack_match = xenium_zstack_match.sort_values(['iou']).drop_duplicates(['mouse_id', 'mask_id_cz'], keep='first')

    xenium_zstack_match.rename(columns={'mask_id_cz': 'cell_id - zstack', 'mask_id_xenium': 'cell_id - xenium'}, inplace=True)
    xenium_zstack_match.sort_values(['mouse_id', 'xenium_section', 'cell_id - zstack'], inplace=True)
    xenium_zstack_match = xenium_zstack_match[['mouse_id', 'xenium_section', 'cell_id - zstack', 'cell_id - xenium']]
    xenium_zstack_match

    ophys_xenium_match = xenium_zstack_match.merge(ophys_zstack_match, on=['mouse_id', 'cell_id - zstack'], how='inner')
    ophys_xenium_match = ophys_xenium_match.drop_duplicates(subset=['mouse_id', 'xenium_section','cell_id - xenium'])
    ophys_xenium_match.insert(0, 'unique_id', ophys_xenium_match.apply(lambda row:( f"{row['mouse_id']}_{row['xenium_section']}_{row['cell_id - zstack']}"), axis=1))
    ophys_xenium_match


,mouse_id,plane,cell_id - zstack,x_loc,y_loc,z_loc,cell_id - session_1,cell_id - session_2,cell_id - session_3
1,778174,VISp_0,484,325.230673,227.530707,100.700193,3,6,2
2,778174,VISp_0,666,245.603893,232.339118,108.792448,6,11,1
4,778174,VISp_0,559,334.640301,237.226141,103.892376,8,15,3
5,778174,VISp_0,560,453.713621,233.807309,103.837431,9,14,4
9,778174,VISp_0,604,482.609598,241.273671,101.070039,14,18,8
...,...,...,...,...,...,...,...,...,...
288,797371,VISp_7,14693,187.539065,618.405034,358.530566,293,301,319
290,797371,VISp_7,14856,378.301977,627.428258,365.104319,295,305,323
291,797371,VISp_7,14518,225.483222,633.090686,360.511281,296,309,325
292,797371,VISp_7,14762,390.919254,634.056262,368.023963,297,307,326


In [ ]:

for mouse_id in mouse_ids:
    xenium_zstack_match = []
    xenium_zstack_path = DATA_DIR / data_info[mouse_id]['coregistration']['xenium_zstack'] / "cell_matching"
    cell_matching_files =   list(xenium_zstack_path.glob("*.csv"))
    for file in cell_matching_files:
        df = pd.read_csv(file)
        df['xenium_section'] = int(file.stem.split('_')[1])
        df['mouse_id'] = mouse_id
        mask_cz_counts = df.groupby('mask_id_cz')['mask_id_xenium'].nunique()
        df = df[df['iou'] > 0.4]
        duplicate_mask_cz = mask_cz_counts[mask_cz_counts > 1].index
        df = df[~df['mask_id_cz'].isin(duplicate_mask_cz)]
        xenium_zstack_match.append(df)
    
    xenium_zstack_match = pd.concat(xenium_zstack_match, ignore_index=True)
    xenium_zstack_match = xenium_zstack_match.sort_values(['iou']).drop_duplicates(['mouse_id', 'mask_id_cz'], keep='first')

    xenium_zstack_match.rename(columns={'mask_id_cz': 'cell_id - zstack', 'mask_id_xenium': 'cell_id - xenium'}, inplace=True)
    xenium_zstack_match.sort_values(['mouse_id', 'xenium_section', 'cell_id - zstack'], inplace=True)
    xenium_zstack_match = xenium_zstack_match[['mouse_id', 'xenium_section', 'cell_id - zstack', 'cell_id - xenium']]
    xenium_zstack_match

,mouse_id,xenium_section,cell_id - zstack,cell_id - xenium
6251,778174,4,385,12406
6240,778174,4,386,9430
6275,778174,4,388,12804
6239,778174,4,389,9161
6238,778174,4,393,7738
...,...,...,...,...
30099,797371,22,15973,2199
30100,797371,22,15989,2200
30337,797371,22,16002,16093
30134,797371,22,16003,7920


# unifed table connecting ophys sessions and xenium

In [6]:
ophys_xenium_match = xenium_zstack_match.merge(ophys_zstack_match, on=['mouse_id', 'cell_id - zstack'], how='inner')
ophys_xenium_match = ophys_xenium_match.drop_duplicates(subset=['mouse_id', 'xenium_section','cell_id - xenium'])
ophys_xenium_match.insert(0, 'unique_id', ophys_xenium_match.apply(lambda row:( f"{row['mouse_id']}_{row['xenium_section']}_{row['cell_id - zstack']}"), axis=1))
ophys_xenium_match

,unique_id,mouse_id,xenium_section,cell_id - zstack,cell_id - xenium,plane,x_loc,y_loc,z_loc,cell_id - session_1,cell_id - session_2,cell_id - session_3
0,778174_4_443,778174,4,443,12305,VISp_0,441.997981,599.909892,91.860892,125,130,117
1,778174_4_449,778174,4,449,12751,VISp_0,483.903570,498.306716,92.694130,86,94,78
2,778174_4_470,778174,4,470,12715,VISp_0,273.799161,270.955895,100.833692,21,27,15
3,778174_4_484,778174,4,484,12795,VISp_0,325.230673,227.530707,100.700193,3,6,2
4,778174_4_488,778174,4,488,12728,VISp_0,457.797195,506.247175,97.006697,89,98,80
...,...,...,...,...,...,...,...,...,...,...,...,...
4026,797371_22_14972,797371,22,14972,10704,VISp_7,565.533513,571.595954,376.737694,265,271,290
4027,797371_22_14977,797371,22,14977,7985,VISp_7,516.286434,634.210836,370.939941,298,306,327
4028,797371_22_15069,797371,22,15069,10679,VISp_7,629.980086,488.764378,379.938797,223,214,234
4029,797371_22_15076,797371,22,15076,7991,VISp_7,496.720414,602.781479,376.309866,279,290,307


In [ ]:
ophys_xenium_match.to_csv('/root/capsule/scratch/ophys_xenium_match.csv', index = False)